In [ ]:
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix, matthews_corrcoef
from PIL import Image
from tqdm.notebook import tqdm

In [ ]:
class VinDrMLODataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        # Mapeamento Binário (BI-RADS 1, 2, 3 = Benigno(0) | BI-RADS 4, 5 = Maligno(1))
        self.label_map = {'BI-RADS 1': 0,
                          'BI-RADS 2': 0,
                          'BI-RADS 3': 0,
                          'BI-RADS 4': 1,
                          'BI-RADS 5': 1
                          }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Caminho do DICOM
        img_path = f"{self.root_dir}/{row['study_id']}/{row['image_id']}.dicom"
        
        # Leitura e Normalização do DICOM
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # Normalização Min-Max para a imagem médica de 16 bits
        pixel_array = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        pixel_array_16bit = (pixel_array * 65535).astype(np.uint16)

        # 2. Inicialização e Aplicação do CLAHE (ajuste o clipLimit se necessário)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        pixel_array_clahe = clahe.apply(pixel_array_16bit)

        # 3. Redução para 8-bits
        pixel_array_8bit = (pixel_array_clahe / 256).astype(np.uint8)
        
        # Converte para imagem PIL em RGB (necessário para os pesos da ResNet)
        image = Image.fromarray(pixel_array_8bit).convert('RGB')
        
        # =========================================================
        # ALINHAMENTO DA MAMA (O Pulo do Gato)
        # =========================================================
        # Como confirmado, lateralidade é sempre 'L' ou 'R'
        lateralidade = row['laterality'] 
        
        # Espelha a mama direita para que todas fiquem orientadas como a esquerda
        if lateralidade == 'R':
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Pega a classe e converte para Binário
        label = self.label_map[row['breast_birads']]
        
        # Aplica Transformações (Tensor, Resize, Normalize...)
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# --- Transformações (Atenção: NÃO há RandomHorizontalFlip aqui!) ---
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(10), # Apenas rotação leve para augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Carregamento e Preparação do CSV ---
csv_path = "/backup/lucas/datasets/vindr-mammo/breast-level_annotations.csv"
images_dir = "/backup/lucas/datasets/vindr-mammo/images"

df_completo = pd.read_csv(csv_path)

# Filtra apenas MLO (Verifique se no CSV chama 'view' ou 'view_position')
df_mlo = df_completo[df_completo['view_position'] == 'MLO'].copy()

# Remove possíveis linhas sem BI-RADS anotado, se houver
df_mlo = df_mlo.dropna(subset=['breast_birads'])

# --- Split sem Data Leakage (por study_id) ---
# 1º Passo: 80% Treino / 20% Temporário
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df_mlo, groups=df_mlo['study_id']))

df_train = df_mlo.iloc[train_idx]
df_temp = df_mlo.iloc[temp_idx]

# 2º Passo: Divide os 20% em 10% Validação e 10% Teste
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['study_id']))

df_val = df_temp.iloc[val_idx]
df_test = df_temp.iloc[test_idx]

print(f"Total MLO: {len(df_mlo)} | Treino: {len(df_train)} | Val: {len(df_val)} | Teste: {len(df_test)}")

# --- DataLoaders ---
BATCH_SIZE = 16 # Ajuste para 8 ou 4 se tiver erro de falta de memória de vídeo (OOM)

# IMPORTANTE: Garanta que as transformações (train_transform e val_transform) 
# estejam definidas corretamente acima desta linha, conforme o padrão da sua ResNet.
train_dataset = VinDrMLODataset(dataframe=df_train, root_dir=images_dir, transform=train_transform)
val_dataset = VinDrMLODataset(dataframe=df_val, root_dir=images_dir, transform=val_transform)
test_dataset = VinDrMLODataset(dataframe=df_test, root_dir=images_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

Total Imagens MLO: 9999 | Treino: 7999 | Validação: 2000


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Modelo ResNet50 ---
model = models.resnet50(weights='IMAGENET1K_V1')

# Troca a última camada para 2 classes (Benigno vs Maligno)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# --- Pesos das Classes para o Desbalanceamento ---
# Casos malignos são minoria. Damos um peso maior para a Classe 1 (Maligno)
# Exemplo: Peso 1.0 para Benigno e 8.0 para Maligno. (Ajuste se precisar de mais sensibilidade)
weights = torch.tensor([1.0, 8.0]).to(device) 

criterion = nn.CrossEntropyLoss(weight=weights)

# Optimizer com um Learning Rate baixo, ideal para transfer learning
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
num_epochs = 20
best_auc = 0.0

for epoch in range(num_epochs):
    print(f"\n--- Época {epoch+1}/{num_epochs} ---")
    
    # ==================================
    # TREINAMENTO
    # ==================================
    model.train()
    train_loss = 0.0
    
    loop_treino = tqdm(train_loader, desc="Treinamento", leave=False)
    for images, labels in loop_treino:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # ==================================
    # VALIDAÇÃO
    # ==================================
    model.eval()
    val_loss = 0.0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        loop_val = tqdm(val_loader, desc="Validação", leave=False)
        for images, labels in loop_val:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            # Probabilidade da classe 1 (Maligno) para calcular o AUC
            probs = F.softmax(outputs, dim=1)[:, 1]
            
            # Previsão direta pegando a classe de maior probabilidade
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # ==================================
    # CÁLCULO DAS MÉTRICAS
    # ==================================
    try:
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    except ValueError:
        tn = fp = fn = tp = 0
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.0
        
    # Novo cálculo do MCC
    try:
        mcc = matthews_corrcoef(all_labels, all_preds)
    except ValueError:
        mcc = 0.0
    
    print(f"Loss Treino: {train_loss:.4f} | Loss Validação: {val_loss:.4f}")
    print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
    print(f"Sensibilidade (Recall): {sensitivity:.4f}")
    print(f"Especificidade:       {specificity:.4f}")
    print(f"AUC-ROC:              {auc:.4f}")
    print(f"MCC:                  {mcc:.4f}")
    
    # ==================================
    # SALVAR MELHOR MODELO
    # ==================================
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'resnet_vindr_mlo_binario.pth')
        print(f"🔥 Novo melhor modelo salvo! (AUC: {best_auc:.4f})")


--- Época 1/20 ---


Loss Treino: 0.5600 | Loss Validação: 0.4643
Matriz de Confusão -> TP:5 | FN:84 | TN:1900 | FP:11
Sensibilidade (Recall): 0.0562
Especificidade:       0.9942
AUC-ROC:              0.6557
🔥 Novo melhor modelo salvo! (AUC: 0.6557)

--- Época 2/20 ---


Loss Treino: 0.5406 | Loss Validação: 0.4722
Matriz de Confusão -> TP:20 | FN:69 | TN:1867 | FP:44
Sensibilidade (Recall): 0.2247
Especificidade:       0.9770
AUC-ROC:              0.6588
🔥 Novo melhor modelo salvo! (AUC: 0.6588)

--- Época 3/20 ---


Loss Treino: 0.5397 | Loss Validação: 0.4933
Matriz de Confusão -> TP:3 | FN:86 | TN:1905 | FP:6
Sensibilidade (Recall): 0.0337
Especificidade:       0.9969
AUC-ROC:              0.5972

--- Época 4/20 ---


Loss Treino: 0.5345 | Loss Validação: 0.4722
Matriz de Confusão -> TP:24 | FN:65 | TN:1853 | FP:58
Sensibilidade (Recall): 0.2697
Especificidade:       0.9696
AUC-ROC:              0.6268

--- Época 5/20 ---


...

Loss Treino: 0.4989 | Loss Validação: 0.4458
Matriz de Confusão -> TP:24 | FN:65 | TN:1848 | FP:63
Sensibilidade (Recall): 0.2697
Especificidade:       0.9670
AUC-ROC:              0.6951
🔥 Novo melhor modelo salvo! (AUC: 0.6951)

--- Época 17/20 ---


Loss Treino: 0.4770 | Loss Validação: 0.5043
Matriz de Confusão -> TP:33 | FN:56 | TN:1745 | FP:166
Sensibilidade (Recall): 0.3708
Especificidade:       0.9131
AUC-ROC:              0.6757

--- Época 18/20 ---


Loss Treino: 0.4778 | Loss Validação: 0.4638
Matriz de Confusão -> TP:17 | FN:72 | TN:1870 | FP:41
Sensibilidade (Recall): 0.1910
Especificidade:       0.9785
AUC-ROC:              0.7021
🔥 Novo melhor modelo salvo! (AUC: 0.7021)

--- Época 19/20 ---


Loss Treino: 0.4714 | Loss Validação: 0.4641
Matriz de Confusão -> TP:22 | FN:67 | TN:1870 | FP:41
Sensibilidade (Recall): 0.2472
Especificidade:       0.9785
AUC-ROC:              0.6881

--- Época 20/20 ---


Loss Treino: 0.4578 | Loss Validação: 0.4627
Matriz de Confusão -> TP:24 | FN:65 | TN:1872 | FP:39
Sensibilidade (Recall): 0.2697
Especificidade:       0.9796
AUC-ROC:              0.6855


In [ ]:
# ==================================
# AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# ==================================

# Carrega os pesos do melhor modelo salvo
model.load_state_dict(torch.load('resnet_vindr_mlo_binario.pth'))
model.eval()

test_loss = 0.0
all_preds_test = []
all_labels_test = []
all_probs_test = []

with torch.no_grad():
    loop_test = tqdm(test_loader, desc="Teste Final ResNet", leave=False)
    for images, labels in loop_test:
        images = images.to(device)
        labels = labels.to(device).long()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        
        # Probabilidade e Previsão (sem threshold manual)
        probs = F.softmax(outputs, dim=1)[:, 1]
        _, preds = torch.max(outputs, 1)
        
        all_labels_test.extend(labels.cpu().numpy())
        all_preds_test.extend(preds.cpu().numpy())
        all_probs_test.extend(probs.cpu().numpy())

test_loss = test_loss / len(test_loader.dataset)

# Cálculo das métricas finais
try:
    tn, fp, fn, tp = confusion_matrix(all_labels_test, all_preds_test, labels=[0, 1]).ravel()
except ValueError:
    tn = fp = fn = tp = 0

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auc = roc_auc_score(all_labels_test, all_probs_test) if len(np.unique(all_labels_test)) > 1 else 0.0
mcc = matthews_corrcoef(all_labels_test, all_preds_test) if len(np.unique(all_labels_test)) > 1 else 0.0

print("\n" + "="*40)
print("🏆 RESULTADOS FINAIS DA RESNET (TESTE) 🏆")
print("="*40)
print(f"Loss Teste:           {test_loss:.4f}")
print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
print(f"Sensibilidade (Recall): {sensitivity:.4f}")
print(f"Especificidade:       {specificity:.4f}")
print(f"AUC-ROC:              {auc:.4f}")
print(f"MCC:                  {mcc:.4f}")
print("="*40)